# 03 - Correlation Analysis

**Question:** How are genomic traits associated, once arithmetic (non-biological)
relationships are removed?

All values are published genetic evaluations representing predicted genetic merit. Most are genomic evaluations, but other proof sources are present. The public repo ships only
`data/sample_synthetic.csv` (structure only); these findings reproduce on the real
export.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd, numpy as np
from scipy import stats
hits=[p for base in [Path("."),Path(".."),Path("../..")] if base.exists()
      for p in base.rglob("Master_Database*.xlsx")]
df=pd.read_excel(hits[0],sheet_name="Data",header=1).dropna(how="all") if hits else pd.read_csv("../data/sample_synthetic.csv")
df.columns=[c.strip() for c in df.columns]
df["year"]=pd.to_datetime(df["Birth Date"],errors="coerce").dt.year
for c in ["Milk (kg)","Fat (kg)","%Fat","Prot (kg)","%Prot","STA","BD","HFE","CW","BMR","HL","ENVIRO","Conf","MS","FL","DS","RU"]:
    if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
print("Loaded",df.shape)

Loaded (668, 139)


## 1. Redundancy: composites vs their own components

A composite trait (`Conf`) is built from its component traits (MS, FL, DS, RU), so a
high correlation between them is arithmetic, not biology. The same holds for production:
component percentages are functions of yields and volume. VIF quantifies this.

In [2]:
def vif_table(cols):
    X=df[cols].dropna(); Xn=(X-X.mean())/X.std(); out={}
    for c in cols:
        y=Xn[c].values; Z=np.column_stack([np.ones(len(Xn)),Xn.drop(columns=[c]).values])
        b,*_=np.linalg.lstsq(Z,y,rcond=None); r2=1-((y-Z@b)**2).sum()/((y-y.mean())**2).sum()
        out[c]=1/(1-r2)
    return out
print("Production block VIF (>10 = severe collinearity):")
for k,v in sorted(vif_table(["Milk (kg)","Fat (kg)","%Fat","Prot (kg)","%Prot"]).items(),key=lambda x:-x[1]):
    print(f"  {k:10s} VIF = {v:7.1f}")

Production block VIF (>10 = severe collinearity):
  %Fat       VIF =   961.2
  Fat (kg)   VIF =   856.7
  Prot (kg)  VIF =   519.4
  Milk (kg)  VIF =   412.0
  %Prot      VIF =   389.9


**Result.** %Fat has VIF = 961, Fat 857, Prot 519, Milk 412: these are near-linearly
dependent because a component percentage is a function of component yield and milk
volume. They are reported in separate blocks and never treated as independent evidence
of one another. `Conf` is likewise excluded from any matrix containing MS, FL, DS or RU.

## 2. Mechanism test: does BMR mediate the Stature-Fat association?

The intuitive hypothesis: smaller cows save maintenance energy (captured by BMR) and
redirect it into fat. This is a formal mediation claim, so it is tested as one:
path a (Stature -> BMR), path b (BMR -> Fat controlling Stature), the indirect effect
a*b with a bootstrap confidence interval, and the direct and total effects.

In [3]:
import statsmodels.formula.api as smf
d=df[["STA","Fat (kg)","BMR","year","Sire Reg Number"]].dropna().rename(columns={"Fat (kg)":"Fat"})
d["SireID"]=d["Sire Reg Number"].astype(str)
# paths adjusted for birth year
a=smf.ols("BMR ~ STA + year",data=d).fit().params["STA"]
mfit=smf.ols("Fat ~ STA + BMR + year",data=d).fit()
b=mfit.params["BMR"]; cprime=mfit.params["STA"]
ctot=smf.ols("Fat ~ STA + year",data=d).fit().params["STA"]
# CLUSTER bootstrap: resample whole sire families (respects non-independence)
sires=d["SireID"].unique(); by={s:d[d.SireID==s] for s in sires}
rng=np.random.default_rng(42); ind=[]
for _ in range(2000):
    pick=rng.choice(sires,len(sires),replace=True)
    bs=pd.concat([by[s] for s in pick],ignore_index=True)
    aa=smf.ols("BMR ~ STA + year",data=bs).fit().params["STA"]
    bb=smf.ols("Fat ~ STA + BMR + year",data=bs).fit().params["BMR"]
    ind.append(aa*bb)
lo,hi=np.percentile(ind,[2.5,97.5])
print(f"n = {len(d)}, sires = {len(sires)} (cluster bootstrap, adjusted for birth year)")
print(f"a  (Stature -> BMR)       = {a:+.3f}")
print(f"b  (BMR -> Fat | Stature) = {b:+.3f}")
print(f"indirect a*b              = {a*b:+.3f}  95% cluster-bootstrap CI [{lo:+.3f}, {hi:+.3f}]")
print(f"direct c'                 = {cprime:+.3f}   total c = {ctot:+.3f}")
print(f"\nindirect CI includes 0: {lo<0<hi}")
print("-> The data do NOT support BMR as the mediator of the Stature-Fat association")
print("   under this model. This does not rule out energy partitioning physiologically;")
print("   it rules out BMR as the mediating variable here.")

n = 666, sires = 212 (cluster bootstrap, adjusted for birth year)
a  (Stature -> BMR)       = -0.505
b  (BMR -> Fat | Stature) = -0.446
indirect a*b              = +0.225  95% cluster-bootstrap CI [-0.326, +0.771]
direct c'                 = -2.736   total c = -2.510

indirect CI includes 0: True
-> The data do NOT support BMR as the mediator of the Stature-Fat association
   under this model. This does not rule out energy partitioning physiologically;
   it rules out BMR as the mediating variable here.


**Result (cluster-bootstrap mediation, sire-resampled, year-adjusted).** With sire
families resampled as clusters and birth year controlled, the indirect effect through BMR
is +0.225 with a 95% CI of [-0.326, +0.771], which **includes zero**. The data therefore
**do not support BMR as the mediator** of the negative Stature-Fat association under this
model.

This is a statement about BMR as a variable in observational genetic evaluations, not a
refutation of energy-partitioning physiology in general, which these data cannot address.
The Stature-Fat association is real; this analysis shows only that BMR does not account
for it here.

## 3. Body-size correlations with functional evaluations

Composite body size (z-mean of Stature, Body Depth, Height at Front End, Chest Width)
correlated against functional and economic evaluations, on all animals.

In [4]:
z=df[["STA","BD","HFE","CW"]].apply(lambda x:(x-x.mean())/x.std())
df["SIZE"]=z.mean(axis=1)
for c in ["Milk (kg)","Fat (kg)","HL","BMR","ENVIRO"]:
    s=df[["SIZE",c]].dropna(); r,p=stats.pearsonr(s["SIZE"],s[c])
    print(f"  SIZE vs {c:10s} r={r:+.3f}  p={p:.2e}  n={len(s)}")

  SIZE vs Milk (kg)  r=+0.011  p=7.76e-01  n=668
  SIZE vs Fat (kg)   r=-0.210  p=4.08e-08  n=668
  SIZE vs HL         r=-0.424  p=1.77e-30  n=668
  SIZE vs BMR        r=-0.654  p=8.82e-83  n=668
  SIZE vs ENVIRO     r=-0.448  p=2.92e-34  n=668


**Result.** SIZE vs Milk r = +0.01 (ns), vs Fat -0.21, vs Herd Life -0.42, vs Body
Maintenance -0.65, vs Environmental Impact -0.45. No milk association; unfavourable
associations with longevity, maintenance and environment. These are associations within
one herd, not causal effects.